# Диалоговые модели: decoder-only и encoder-decoder

Два независимых эксперимента. В каждом своя модель и свой датасет. Общие только функции предобработки и подсчёта метрик.

Для каждой архитектуры:

1. Выбрать модель: https://huggingface.co/models
2. Выбрать датасет: https://huggingface.co/datasets
3. Проверить модель в сценарии, близком к её датасету, в zero-shot режиме.
4. Посчитать BLEU и METEOR исходной модели на валидационном сабсете.
5. Дообучить модель на обучающем сабсете.
6. Проверить модель в том же сценарии после дообучения.
7. Посчитать BLEU и METEOR дообученной модели на валидационном сабсете.
8. Сравнить метрики дообученной и исходной моделей.

Диалоговый датасет - [teknium/OpenHermes-2.5](https://huggingface.co/datasets/teknium/OpenHermes-2.5).

- Decoder-only: `HuggingFaceTB/SmolLM2-360M-Instruct`
- Encoder-decoder: `google/flan-t5-base`

Для каждой модели считаются BLEU и METEOR до обучения, модель дообучается, те же метрики считаются снова, затем сравниваются.


Обучающая выборка - 2000 пар, валидация - 50. Лимиты задаются константами `MAX_TRAIN_SAMPLES` и `MAX_VAL_SAMPLES`.


## Установка

Обновление transformers в Colab

In [ ]:
!uv pip install -q -U transformers datasets trl evaluate accelerate bitsandbytes nltk sentencepiece


## Настройки

Веса загружаются и обучаются в bf16. Adam 8-bit и gradient checkpointing держат состояния оптимизатора encoder-decoder модели. Чекпоинты на диск не пишутся: сравнение идёт по модели в памяти.


## Импорты

In [ ]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

import gc

import datasets
import evaluate
import nltk
import pandas as pd
import torch
import transformers
import trl
from datasets import Dataset, DatasetDict, load_dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("trl", trl.__version__)

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

print("GPU:", torch.cuda.get_device_name(0), "(physical cuda:3)")


transformers 5.20.2
datasets 5.1.0
trl 1.16.0


GPU: NVIDIA H100 80GB HBM3 (physical cuda:3)


## Константы

In [ ]:
DATASET_NAME = "teknium/OpenHermes-2.5"
DECODER_MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
ENCDEC_MODEL = "google/flan-t5-base"

SEED = 42
MAX_TRAIN_SAMPLES = 2000
MAX_VAL_SAMPLES = 50
MAX_HISTORY_TURNS = 6
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 64
MAX_NEW_TOKENS = 50
N_QUAL_EXAMPLES = 3

EPOCHS = 3
LR = 2e-5
BATCH_SIZE = 32
GRAD_ACCUM = 1
LOGGING_STEPS = 10

DEVICE = "cuda"

set_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA не найдена. В Colab: Runtime → Change runtime type → T4 GPU, затем перезапустите сессию."
    )

## Данные

[teknium/OpenHermes-2.5](https://huggingface.co/datasets/teknium/OpenHermes-2.5) - диалоги с нескольими шагами обсуждения. Реплики лежат в `conversations`: `human` / `user` - пользователь, `gpt` / `assistant` - ответ, `system` - системная инструкция. У датасета один сплит `train`, валидация отрезается от него.

Для каждой реплики ассистента цель - её текст. В контекст входят не больше шести предыдущих реплик. Flan-T5 видит обычный текст `User` / `Assistant`. SmolLM2-Instruct получает тот же диалог через свой chat template.


In [ ]:
ROLE_MAP = {
    "human": "user",
    "user": "user",
    "gpt": "assistant",
    "assistant": "assistant",
    "system": "system",
}
SPEAKER = {"system": "System", "user": "User", "assistant": "Assistant"}


def turns_from_conversation(conversation):
    turns = []
    for turn in conversation or []:
        if isinstance(turn, str):
            continue
        role = ROLE_MAP.get(str(turn.get("from") or turn.get("role") or "").strip().lower())
        text = str(turn.get("value") or turn.get("content") or "").strip()
        if role and text:
            turns.append({"role": role, "content": text})
    return turns


def plain_prompt(messages):
    lines = [f"{SPEAKER[message['role']]}: {message['content']}" for message in messages]
    return "\n".join(lines) + "\nAssistant:"


def fit_messages(messages, target):
    # SFT обрезает последовательность с конца, поэтому контекст укорачивается, а ответ остаётся.
    kept = list(messages[-MAX_HISTORY_TURNS:])
    while True:
        fits = (len(plain_prompt(kept)) + 1 + len(target)) / 3 <= MAX_SOURCE_LENGTH
        if len(kept) <= 1 or fits:
            return kept
        drop_at = 1 if kept[0]["role"] == "system" and len(kept) > 2 else 0
        kept.pop(drop_at)


def build_pairs(split):
    rows = []
    for row in split:
        history = []
        for turn in turns_from_conversation(row.get("conversations") or row.get("messages")):
            if turn["role"] == "assistant":
                if history:
                    context = fit_messages(history, turn["content"])
                    rows.append(
                        {
                            "prompt": plain_prompt(context),
                            "target": turn["content"],
                            "messages": context,
                        }
                    )
            history.append(turn)
    if not rows:
        raise ValueError("Из OpenHermes не собраны пары «контекст - ответ».")
    return Dataset.from_list(rows)


raw = load_dataset(DATASET_NAME, split="train")
raw = raw.shuffle(seed=SEED)
needed = MAX_TRAIN_SAMPLES + MAX_VAL_SAMPLES
chunk = raw.select(range(min(len(raw), needed * 3)))
pairs = build_pairs(chunk).shuffle(seed=SEED)
if len(pairs) < needed:
    raise ValueError(f"Нужно минимум {needed} пар, собрано {len(pairs)}.")
train_subset = pairs.select(range(MAX_TRAIN_SAMPLES))
val_subset = pairs.select(range(MAX_TRAIN_SAMPLES, needed))
print(raw)
print("train pairs:", len(train_subset), "val pairs:", len(val_subset))
print(train_subset[0]["prompt"])
print("target:", train_subset[0]["target"])


README.md:   0%|          | 0.00/4.75k [00:00<?, ?B/s]

openhermes2_5.json: reconstructing file:   0%|          |  0.00B / 1.94GB            

openhermes2_5.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1001551 [00:00<?, ? examples/s]

Dataset({
    features: ['custom_instruction', 'topic', 'model_name', 'model', 'skip_prompt_formatting', 'category', 'conversations', 'views', 'language', 'id', 'title', 'idx', 'hash', 'avatarUrl', 'system_prompt', 'source'],
    num_rows: 1001551
})
train pairs: 2000 val pairs: 50
User: Question: Process: - Air is brought in through the mouth - Passes through the lungs - And into the bronchial tissue - The carbon dioxide is removed - The lungs bring the oxygen to the rest of the body.  Question: suppose there is less sand being produced happens, how will it affect if lungs stop working properly and at a lower rate.  How does the supposed perturbation influence the second effect mentioned. Answer by more, less or no effect
Answer:
Assistant:
target: Less effect


## Метрики

BLEU и METEOR считаются на `val_subset`. Генерация жадная, не длиннее 50 новых токенов, по одному примеру, чтобы кэш декодера не выходил за 16 ГБ.

У decoder-only в метрику попадает только текст после промпта. У encoder-decoder `generate` сразу возвращает ответ. Печатаются первые три примера.


In [ ]:
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")


def training_kwargs(output_dir):
    return dict(
        output_dir=output_dir,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LR,
        num_train_epochs=EPOCHS,
        logging_steps=LOGGING_STEPS,
        save_strategy="no",
        fp16=False,
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="adamw_8bit",
        report_to="none",
        dataloader_num_workers=0,
        seed=SEED,
    )


def evaluate_generation(model, tokenizer, rows, architecture):
    model.eval()
    model.config.use_cache = True
    predictions = []
    references = []

    for index, sample in enumerate(tqdm(rows, desc="generate")):
        inputs = tokenizer(sample["prompt"], return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        if architecture == "decoder":
            generated_ids = outputs[0][inputs["input_ids"].shape[1] :]
        else:
            generated_ids = outputs[0]
        generated_text = tokenizer.decode(generated_ids.tolist(), skip_special_tokens=True).strip()
        predictions.append(generated_text)
        references.append([sample["target"]])
        if index < N_QUAL_EXAMPLES:
            print(f"--- {index + 1} ---")
            print(sample["prompt"])
            print("Reference:", sample["target"])
            print("Model:", generated_text)

    bleu_score = bleu.compute(predictions=predictions, references=references)
    meteor_score = meteor.compute(predictions=predictions, references=references)
    metrics = {"bleu": bleu_score["bleu"], "meteor": meteor_score["meteor"]}
    print(f"BLEU: {metrics['bleu']:.4f}")
    print(f"METEOR: {metrics['meteor']:.4f}")
    return metrics


def compare_metrics(model_name, before, after):
    frame = pd.DataFrame(
        [
            {
                "model": model_name,
                "metric": metric,
                "before": before[metric],
                "after": after[metric],
                "delta": after[metric] - before[metric],
            }
            for metric in ("bleu", "meteor")
        ]
    )
    print(frame.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
    return frame


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()


[nltk_data] Downloading package wordnet to /home/jovyan/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/jovyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/jovyan/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


## Decoder-only: SmolLM2-360M-Instruct

Промпт и обучающая строка собираются chat template этой модели. `SFTTrainer` учится на колонке `full_text`. Колонка `prompt` в обучение не передаётся: при ней тренер ждёт пару `prompt` / `completion` и не читает `full_text`.

Перед генерацией `use_cache` включается, на время дообучения с gradient checkpointing выключается.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(DECODER_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def format_decoder(example):
    messages = example["messages"]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    full_text = tokenizer.apply_chat_template(
        messages + [{"role": "assistant", "content": example["target"]}],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"prompt": prompt, "target": example["target"], "full_text": full_text}


decoder_train = train_subset.map(format_decoder, remove_columns=train_subset.column_names)
decoder_val = val_subset.map(format_decoder, remove_columns=val_subset.column_names)

model = AutoModelForCausalLM.from_pretrained(DECODER_MODEL, dtype=torch.bfloat16)
model.config.pad_token_id = tokenizer.pad_token_id
model.to(DEVICE)

print("До обучения")
decoder_before = evaluate_generation(model, tokenizer, decoder_val, "decoder")

model.config.use_cache = False
sft_train = decoder_train.select_columns(["full_text"])
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=SFTConfig(
        dataset_text_field="full_text",
        max_length=MAX_SOURCE_LENGTH,
        **training_kwargs("./smollm2_instruct_openhermes"),
    ),
    train_dataset=sft_train,
)
trainer.train()
model = trainer.model

print("\nПосле обучения")
decoder_after = evaluate_generation(model, tokenizer, decoder_val, "decoder")
decoder_comparison = compare_metrics("SmolLM2-360M-Instruct", decoder_before, decoder_after)

del model, tokenizer, trainer, sft_train, decoder_train, decoder_val
free_gpu()


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

generate:   0%|          | 0/50 [00:00<?, ?it/s]

--- 1 ---
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Can you provide an AI solution to perform a substitution cipher on a given string?<|im_end|>
<|im_start|>assistant

Reference: Sure! Here's an AI-based solution to perform a substitution cipher in Python.

```python
def substitution_cipher(plain_text):
    ALPHABET = 'abcdefghijklmnopqrstuvwxyz'
    key = 'zxcvbnmlkjhgfdsaqwertyuiop'
    cipher_text = ''

    for char in plain_text:
        if char == ' ':
            cipher_text += ' '
        else: 
            char_index = ALPHABET.index(char)
            cipher_text += key[char_index]

    return cipher_text
```

In this solution, we define two strings: `ALPHABET` which represents the original alphabet, and `key` which represents the substitution alphabet. The `plain_text` parameter is the input string that we want to encrypt.

We then initialize an empty string `cipher_text` to store the encrypted result. We

--- 2 ---
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Convert the text "3.14" to a number, ensuring that the number is rounded to the nearest hundredth, and the final result must be expressed in scientific notation.<|im_end|>
<|im_start|>assistant

Reference: The number "3.14" rounded to the nearest hundredth is "3.14". Expressed in scientific notation, it would be "3.14 x 10^0".
Model: 3.14 can be converted to a number rounded to the nearest hundredth as follows:

The number 3.14 can be rounded to the nearest hundredth as follows:
3.14

The final result in


--- 3 ---
<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Please write a function in [Java code] to compare two strings and output true if they are identical and false if they are different. The strings are case-sensitive, meaning that uppercase and lowercase letters should be treated as distinct. Here are the two strings to be compared:
String1: "Hello World!"
String2: "Hello world!"
Can you also modify the function to make the comparison case-insensitive, meaning that uppercase and lowercase letters should be treated as equal?<|im_end|>
<|im_start|>assistant

Reference: Sure, here is the function to compare two strings in Java:
```java
public static boolean compareStrings(String str1, String str2) {
    if(str1.equals(str2)) {
        return true;
    }
    else {
        return false;
    }
}
```
To make the comparison case-insensitive, we can modify the function as follows:
```java
public static boolean compareStrin

BLEU: 0.0047
METEOR: 0.1849
                model metric  before  after  delta
SmolLM2-360M-Instruct   bleu  0.0026 0.0047 0.0021
SmolLM2-360M-Instruct meteor  0.1713 0.1849 0.0136


## Encoder-decoder: Flan-T5-base

`SFTTrainer` считает каузальный loss и вызывает `fused_lm_head`. У encoder-decoder этого пути нет, поэтому здесь `Seq2SeqTrainer` с теми же гиперпараметрами. В энкодер подаётся только промпт, метки - ответ.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ENCDEC_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(ENCDEC_MODEL, dtype=torch.bfloat16)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id
model.to(DEVICE)

print("До обучения")
encdec_before = evaluate_generation(model, tokenizer, val_subset, "encoder-decoder")


def tokenize_encdec(batch):
    tokenizer.truncation_side = "left"
    model_inputs = tokenizer(batch["prompt"], truncation=True, max_length=MAX_SOURCE_LENGTH)
    tokenizer.truncation_side = "right"
    labels = tokenizer(text_target=batch["target"], truncation=True, max_length=MAX_TARGET_LENGTH)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train_subset.map(
    tokenize_encdec,
    batched=True,
    remove_columns=train_subset.column_names,
)
model.config.use_cache = False
trainer = Seq2SeqTrainer(
    model=model,
    args=Seq2SeqTrainingArguments(**training_kwargs("./flan_t5_openhermes")),
    train_dataset=tokenized_train,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)
trainer.train()
model = trainer.model

print("\nПосле обучения")
encdec_after = evaluate_generation(model, tokenizer, val_subset, "encoder-decoder")
encdec_comparison = compare_metrics("flan-t5-base", encdec_before, encdec_after)

del model, tokenizer, trainer, tokenized_train
free_gpu()


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


До обучения


generate:   0%|          | 0/50 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (524 > 512). Running this sequence through the model will result in indexing errors


...


После обучения


generate:   0%|          | 0/50 [00:00<?, ?it/s]

--- 1 ---
User: Can you provide an AI solution to perform a substitution cipher on a given string?
Assistant:
Reference: Sure! Here's an AI-based solution to perform a substitution cipher in Python.

```python
def substitution_cipher(plain_text):
    ALPHABET = 'abcdefghijklmnopqrstuvwxyz'
    key = 'zxcvbnmlkjhgfdsaqwertyuiop'
    cipher_text = ''

    for char in plain_text:
        if char == ' ':
            cipher_text += ' '
        else: 
            char_index = ALPHABET.index(char)
            cipher_text += key[char_index]

    return cipher_text
```

In this solution, we define two strings: `ALPHABET` which represents the original alphabet, and `key` which represents the substitution alphabet. The `plain_text` parameter is the input string that we want to encrypt.

We then initialize an empty string `cipher_text` to store the encrypted result. We iterate over each character in `plain_text`, and for each character, we perform the following steps:
- If the character is a space

--- 3 ---
User: Please write a function in [Java code] to compare two strings and output true if they are identical and false if they are different. The strings are case-sensitive, meaning that uppercase and lowercase letters should be treated as distinct. Here are the two strings to be compared:
String1: "Hello World!"
String2: "Hello world!"
Can you also modify the function to make the comparison case-insensitive, meaning that uppercase and lowercase letters should be treated as equal?
Assistant:
Reference: Sure, here is the function to compare two strings in Java:
```java
public static boolean compareStrings(String str1, String str2) {
    if(str1.equals(str2)) {
        return true;
    }
    else {
        return false;
    }
}
```
To make the comparison case-insensitive, we can modify the function as follows:
```java
public static boolean compareStringsIgnoreCase(String str1, String str2) {
    if(str1.equalsIgnoreCase(str2)) {
        return true;
    }
    else {
        return

BLEU: 0.0000
METEOR: 0.1177
       model metric  before  after  delta
flan-t5-base   bleu  0.0000 0.0000 0.0000
flan-t5-base meteor  0.1047 0.1177 0.0130


## Сравнение

BLEU и METEOR обеих моделей до обучения и после. `delta` — на сколько метрика изменилась.


In [ ]:
comparison = pd.concat([decoder_comparison, encdec_comparison], ignore_index=True)
comparison


,model,metric,before,after,delta
0,SmolLM2-360M-Instruct,bleu,2.602367e-03,0.004700,0.002098
1,SmolLM2-360M-Instruct,meteor,1.713018e-01,0.184894,0.013592
2,flan-t5-base,bleu,5.871720e-07,0.000003,0.000002
3,flan-t5-base,meteor,1.047017e-01,0.117656,0.012955


## Вывод

Модели энкодер-декордер справились хуже, но это вызвано задачами, на который она была предобучена.

Также роль играет специфика задачи - одну и ту же задачу по написанию кода и прочего можно решить несколькими способами

LLM-as-a-Judge как метрика
